# 전처리 파이프라인

본 노트북은 CCTV 기반 낙상 감지 시스템을 위한 MoveNet 관절 데이터 추출 및 정제 과정을 담고 있습니다.
전체 파이프라인은 3단계로 구성됩니다.

1. **Step 1:** 원본 영상 전처리 (256x256, 15fps) 및 순수 원시 데이터 추출
2. **Step 2:** 동적 운동학 기반 이상치 차단 및 결측치 대체
3. **Step 3:** 4종 필터(EMA, Kalman, 1-Euro, Butterworth) 적용 및 성능 비교
4. **Step 4:** 낙상 판별용 핵심 물리량(Feature) 추출

In [66]:
# 공통 라이브러리 임포트
import cv2
import tensorflow as tf
import pandas as pd
import numpy as np
import math
from scipy.signal import butter, lfilter
from google.colab import drive

# 전역 설정값
drive.mount('/content/drive')
VIDEO_PATH = '/content/drive/MyDrive/졸업 과제/dataset/source/Y/FY/00007_H_A_FY_C7/00007_H_A_FY_C7.mp4'
MODEL_PATH = '/content/drive/MyDrive/졸업 과제/model/movenet_singlepose_thunder_a100_256_int8.tflite'
TARGET_DIM = 256
TARGET_FPS = 15

KEYPOINT_NAMES = [
    'nose', 'left_eye', 'right_eye', 'left_ear', 'right_ear',
    'left_shoulder', 'right_shoulder', 'left_elbow', 'right_elbow',
    'left_wrist', 'right_wrist', 'left_hip', 'right_hip',
    'left_knee', 'right_knee', 'left_ankle', 'right_ankle'
]

EDGES = [
    (0, 1), (0, 2), (1, 3), (2, 4), (0, 5), (0, 6), (5, 7), (7, 9),
    (6, 8), (8, 10), (5, 6), (5, 11), (6, 12), (11, 12), (11, 13), (13, 15),
    (12, 14), (14, 16)
]

print("환경 설정 완료.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
환경 설정 완료.


## 1. 원본 영상 전처리 및 MoveNet 베이스 데이터 추출
* **목표:** 원본 영상을 `256x256` 해상도와 `15fps`로 Center Crop하여 전처리한 후, MoveNet TFLite 모델을 통해 17개 관절의 위치와 신뢰도를 추출합니다.
* **출력:** `step1_raw_data.csv`, `step1_overlay_raw.mp4`

In [67]:
print("Step 1: 데이터 추출 시작...")

interpreter = tf.lite.Interpreter(model_path=MODEL_PATH)
interpreter.allocate_tensors()
input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()
input_dtype = input_details[0]['dtype']

cap = cv2.VideoCapture(VIDEO_PATH)
orig_fps = cap.get(cv2.CAP_PROP_FPS)
orig_w, orig_h = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
min_d = min(orig_w, orig_h)
sx, sy = (orig_w - min_d) // 2, (orig_h - min_d) // 2
f_skip = max(1, int(round(orig_fps / TARGET_FPS)))

out_vid = cv2.VideoWriter('step1_overlay_raw.mp4', cv2.VideoWriter_fourcc(*'mp4v'), TARGET_FPS, (TARGET_DIM, TARGET_DIM))

raw_data = []
orig_cnt, proc_cnt = 0, 0

while cap.isOpened():
    ret, frame = cap.read()
    if not ret: break
    if orig_cnt % f_skip != 0:
        orig_cnt += 1
        continue

    res_f = cv2.resize(frame[sy:sy+min_d, sx:sx+min_d], (TARGET_DIM, TARGET_DIM))
    input_t = tf.cast(np.expand_dims(cv2.cvtColor(res_f, cv2.COLOR_BGR2RGB), axis=0), dtype=input_dtype)

    interpreter.set_tensor(input_details[0]['index'], input_t)
    interpreter.invoke()
    kps = interpreter.get_tensor(output_details[0]['index'])[0, 0, :, :]
    if output_details[0]['dtype'] != np.float32:
        s, zp = output_details[0]['quantization']
        kps = (kps.astype(np.float32) - zp) * s

    t_sec = orig_cnt / orig_fps
    row = [proc_cnt, t_sec]
    for kp in kps: row.extend([kp[0], kp[1], kp[2]])
    raw_data.append(row)

    # 모든 관절 파란색 표기
    for y, x, score in kps:
        cx, cy = int(x * TARGET_DIM), int(y * TARGET_DIM)
        cv2.circle(res_f, (cx, cy), 3, (255, 0, 0), -1)

    # 모든 뼈대 선 초록색 표기
    for p1, p2 in EDGES:
        y1, x1, s1 = kps[p1]
        y2, x2, s2 = kps[p2]
        cv2.line(res_f, (int(x1*TARGET_DIM), int(y1*TARGET_DIM)), (int(x2*TARGET_DIM), int(y2*TARGET_DIM)), (0, 255, 0), 1)

    out_vid.write(res_f)
    proc_cnt += 1; orig_cnt += 1

cap.release(); out_vid.release()

cols = ['frame', 'timestamp_sec']
for n in KEYPOINT_NAMES: cols.extend([f'{n}_y', f'{n}_x', f'{n}_score'])
df_raw = pd.DataFrame(raw_data, columns=cols)
df_raw.to_csv('step1_raw_data.csv', index=False)
print("Step 1 완료: 'step1_raw_data.csv', 'step1_overlay_raw.mp4' 생성됨.")

Step 1: 데이터 추출 시작...


/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


Step 1 완료: 'step1_raw_data.csv', 'step1_overlay_raw.mp4' 생성됨.


## 2. 이상치 차단, 결측치 보간 및 시각화
* **목표:** 1단계 데이터를 기반으로 객체 오탐을 제거한 후(필터링), 추적을 놓친 구간(NaN)을 선형 보간법으로 부드럽게 채워 넣습니다. 보간이 완료된 연속적인 데이터를 이용해 결과 영상을 생성합니다.
* **출력:** `step2_filtered_data.csv`, `step2_overlay_filtered.mp4`

In [68]:
print("Step 2: 이상치 차단, 결측치 보간(Interpolation) 및 시각화 시작...")

HIGH_THRESH, INITIAL_THRESH, LOW_THRESH = 0.4, 0.2, 0.1
MAX_SPEED_RATIO, MAX_HOLD_FRAMES = 1.2, 5
CONSISTENCY_RADIUS_RATIO = 1.8

df_raw = pd.read_csv('step1_raw_data.csv')

last_valid_pos = {i: None for i in range(17)}
missing_counts = {i: 0 for i in range(17)}
current_torso_length = 0.2
filtered_data = []

# ==========================================
# [Phase 1] 데이터 필터링 (이상치 차단 및 NaN 할당)
# ==========================================
for proc_cnt in range(len(df_raw)):
    row_raw = df_raw.iloc[proc_cnt]
    kps = np.array([[row_raw[f'{n}_y'], row_raw[f'{n}_x'], row_raw[f'{n}_score']] for n in KEYPOINT_NAMES])

    if all(kps[i][2] >= HIGH_THRESH for i in [5, 6, 11, 12]):
        current_torso_length = max(np.linalg.norm((kps[5][:2]+kps[6][:2])/2 - (kps[11][:2]+kps[12][:2])/2), 0.05)

    speed_limit = current_torso_length * MAX_SPEED_RATIO
    consistency_limit = current_torso_length * CONSISTENCY_RADIUS_RATIO

    temp_valid = {}
    for i, (y, x, score) in enumerate(kps):
        if score >= LOW_THRESH:
            if last_valid_pos[i] is not None:
                if np.linalg.norm(np.array([x, y]) - last_valid_pos[i]) <= speed_limit:
                    temp_valid[i] = np.array([x, y])
            else:
                if score >= INITIAL_THRESH:
                    temp_valid[i] = np.array([x, y])

    final_indices = []
    if len(temp_valid) >= 5:
        centroid = np.mean(list(temp_valid.values()), axis=0)
        final_indices = [i for i, pos in temp_valid.items() if np.linalg.norm(pos - centroid) <= consistency_limit]
    else:
        final_indices = list(temp_valid.keys())

    final_kps = []
    for i in range(17):
        if i in final_indices:
            last_valid_pos[i], missing_counts[i] = np.array([kps[i][1], kps[i][0]]), 0
            final_kps.append([kps[i][0], kps[i][1], kps[i][2]])
        elif missing_counts[i] <= MAX_HOLD_FRAMES and last_valid_pos[i] is not None:
            missing_counts[i] += 1
            final_kps.append([last_valid_pos[i][1], last_valid_pos[i][0], kps[i][2]])
        else:
            last_valid_pos[i], missing_counts[i] = None, missing_counts[i]+1
            final_kps.append([np.nan, np.nan, np.nan])

    row_out = [proc_cnt, row_raw['timestamp_sec']]
    for fk in final_kps: row_out.extend(fk)
    filtered_data.append(row_out)

cols = ['frame', 'timestamp_sec']
for n in KEYPOINT_NAMES: cols.extend([f'{n}_y', f'{n}_x', f'{n}_score'])
df_step2 = pd.DataFrame(filtered_data, columns=cols)

# ==========================================
# [Phase 2] 결측치 선형 보간 (Interpolation)
# ==========================================
coord_cols = [f'{n}_{c}' for n in KEYPOINT_NAMES for c in ['x', 'y']]
df_step2[coord_cols] = df_step2[coord_cols].interpolate(method='linear', limit=15, limit_direction='forward')
df_step2[coord_cols] = df_step2[coord_cols].fillna(method='bfill')

df_step2.to_csv('step2_filtered_data.csv', index=False)

# ==========================================
# [Phase 3] 보간된 데이터 기반 영상 시각화
# ==========================================
cap = cv2.VideoCapture(VIDEO_PATH)
orig_fps = cap.get(cv2.CAP_PROP_FPS)
orig_w, orig_h = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
min_d = min(orig_w, orig_h)
sx, sy = (orig_w - min_d) // 2, (orig_h - min_d) // 2
f_skip = max(1, int(round(orig_fps / TARGET_FPS)))

# 코덱을 mp4v로 원복
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out_vid = cv2.VideoWriter('step2_overlay_filtered.mp4', fourcc, TARGET_FPS, (TARGET_DIM, TARGET_DIM))

orig_cnt, proc_cnt = 0, 0
try:
    while cap.isOpened() and proc_cnt < len(df_step2):
        ret, frame = cap.read()
        if not ret: break
        if orig_cnt % f_skip != 0:
            orig_cnt += 1; continue

        res_f = cv2.resize(frame[sy:sy+min_d, sx:sx+min_d], (TARGET_DIM, TARGET_DIM))
        row = df_step2.iloc[proc_cnt]

        kps_viz = {}
        for i, n in enumerate(KEYPOINT_NAMES):
            fx, fy = row[f'{n}_x'], row[f'{n}_y']
            if not pd.isna(fx) and not pd.isna(fy):
                cx, cy = int(fx * TARGET_DIM), int(fy * TARGET_DIM)
                kps_viz[i] = (cx, cy)
                cv2.circle(res_f, (cx, cy), 3, (255, 0, 0), -1)

        for p1, p2 in EDGES:
            if p1 in kps_viz and p2 in kps_viz:
                cv2.line(res_f, kps_viz[p1], kps_viz[p2], (0, 255, 0), 1)

        out_vid.write(res_f)
        proc_cnt += 1; orig_cnt += 1
finally:
    cap.release()
    out_vid.release()

print("Step 2 완료: 보간이 적용된 'step2_filtered_data.csv', 'step2_overlay_filtered.mp4' 생성됨.")

Step 2: 이상치 차단, 결측치 보간(Interpolation) 및 시각화 시작...


/tmp/ipykernel_492/2321365023.py:69: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df_step2[coord_cols] = df_step2[coord_cols].fillna(method='bfill')


Step 2 완료: 보간이 적용된 'step2_filtered_data.csv', 'step2_overlay_filtered.mp4' 생성됨.


## 3. 다중 필터 적용 및 비교
* **목표:** 2단계에서 이상치가 제거된 데이터에 4가지 필터(EMA, Kalman, 1-Euro, Butterworth)를 개별 적용합니다.
* **출력:** 필터별 4개의 CSV 파일 및 4개의 오버레이 영상

In [69]:
print("Step 3: 4종 필터 적용 및 생성 시작...")

class OneEuroFilter:
    def __init__(self, t0, x0, min_cutoff=0.5, beta=1.5, d_cutoff=1.0):
        self.min_cutoff, self.beta, self.d_cutoff = min_cutoff, beta, d_cutoff
        self.x_prev, self.dx_prev, self.t_prev = x0, 0.0, t0
    def alpha(self, t_e, cutoff):
        r = 2 * math.pi * cutoff * t_e
        return r / (r + 1)
    def __call__(self, t, x):
        if pd.isna(x): return x
        t_e = t - self.t_prev
        if t_e <= 0: return x
        dx = (x - self.x_prev) / t_e
        dx_hat = self.alpha(t_e, self.d_cutoff) * dx + (1 - self.alpha(t_e, self.d_cutoff)) * self.dx_prev
        cutoff = self.min_cutoff + self.beta * abs(dx_hat)
        a = self.alpha(t_e, cutoff)
        x_hat = a * x + (1 - a) * self.x_prev
        self.x_prev, self.dx_prev, self.t_prev = x_hat, dx_hat, t
        return x_hat

class SimpleKalman:
    def __init__(self, x0, q=1e-4, r=1e-2):
        self.x, self.P, self.Q, self.R = x0, 1.0, q, r
    def __call__(self, z):
        if pd.isna(z): return z
        self.P += self.Q
        K = self.P / (self.P + self.R)
        self.x += K * (z - self.x)
        self.P *= (1 - K)
        return self.x

def butter_lowpass_filter(data, cutoff=1.5, fs=15, order=2):
    nyq = 0.5 * fs
    normal_cutoff = cutoff / nyq
    b, a = butter(order, normal_cutoff, btype='low', analog=False)
    return lfilter(b, a, data.fillna(method='ffill').fillna(method='bfill').fillna(0))

df_s2 = pd.read_csv('step2_filtered_data.csv')
filter_types = ['EMA', 'Kalman', '1-Euro', 'Butterworth']

for f_type in filter_types:
    print(f" -> {f_type} 필터 처리 중...")
    cap = cv2.VideoCapture(VIDEO_PATH)
    out_vid = cv2.VideoWriter(f'step3_overlay_{f_type}.mp4', cv2.VideoWriter_fourcc(*'mp4v'), TARGET_FPS, (TARGET_DIM, TARGET_DIM))

    prev_f_x, prev_f_y = {i: None for i in range(17)}, {i: None for i in range(17)}
    filters_x, filters_y = {}, {}

    df_f = df_s2.copy()
    if f_type == 'Butterworth':
        for n in KEYPOINT_NAMES:
            df_f[f'{n}_x'] = butter_lowpass_filter(df_s2[f'{n}_x'])
            df_f[f'{n}_y'] = butter_lowpass_filter(df_s2[f'{n}_y'])

    filtered_csv_data = []
    row_idx, orig_cnt = 0, 0

    while cap.isOpened() and row_idx < len(df_s2):
        ret, frame = cap.read()
        if not ret: break
        if orig_cnt % f_skip != 0:
            orig_cnt += 1; continue

        res_f = cv2.resize(frame[sy:sy+min_d, sx:sx+min_d], (TARGET_DIM, TARGET_DIM))
        row = df_s2.iloc[row_idx]
        kps_viz = {}
        row_out = [row['frame'], row['timestamp_sec']]

        for i, n in enumerate(KEYPOINT_NAMES):
            rx, ry, score = row[f'{n}_x'], row[f'{n}_y'], row[f'{n}_score']

            if pd.isna(rx):
                prev_f_x[i] = prev_f_y[i] = None
                row_out.extend([np.nan, np.nan, score])
                continue

            if f_type == 'EMA':
                fx, fy = (score*rx + (1-score)*prev_f_x[i], score*ry + (1-score)*prev_f_y[i]) if prev_f_x[i] is not None else (rx, ry)
            elif f_type == 'Kalman':
                if i not in filters_x: filters_x[i], filters_y[i] = SimpleKalman(rx), SimpleKalman(ry)
                fx, fy = filters_x[i](rx), filters_y[i](ry)
            elif f_type == '1-Euro':
                if i not in filters_x: filters_x[i], filters_y[i] = OneEuroFilter(row['timestamp_sec'], rx), OneEuroFilter(row['timestamp_sec'], ry)
                fx, fy = filters_x[i](row['timestamp_sec'], rx), filters_y[i](row['timestamp_sec'], ry)
            elif f_type == 'Butterworth':
                fx, fy = df_f.iloc[row_idx][f'{n}_x'], df_f.iloc[row_idx][f'{n}_y']

            row_out.extend([fy, fx, score])

            if not pd.isna(fx):
                cx, cy = int(fx*TARGET_DIM), int(fy*TARGET_DIM)
                kps_viz[i] = (cx, cy)
                cv2.circle(res_f, (cx, cy), 3, (255, 0, 0), -1)
                prev_f_x[i], prev_f_y[i] = fx, fy

        for p1, p2 in EDGES:
            if p1 in kps_viz and p2 in kps_viz: cv2.line(res_f, kps_viz[p1], kps_viz[p2], (0, 255, 0), 1)

        cv2.putText(res_f, f_type, (10, 30), 1, 2, (255, 255, 255), 2)
        out_vid.write(res_f)
        filtered_csv_data.append(row_out)

        row_idx += 1; orig_cnt += 1

    cap.release(); out_vid.release()
    pd.DataFrame(filtered_csv_data, columns=cols).to_csv(f'step3_filtered_{f_type}.csv', index=False)

print("모든 파이프라인 처리가 완료되었습니다.")

Step 3: 4종 필터 적용 및 생성 시작...
 -> EMA 필터 처리 중...
 -> Kalman 필터 처리 중...
 -> 1-Euro 필터 처리 중...
 -> Butterworth 필터 처리 중...


/tmp/ipykernel_492/3705102711.py:37: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  return lfilter(b, a, data.fillna(method='ffill').fillna(method='bfill').fillna(0))


모든 파이프라인 처리가 완료되었습니다.


## 4. 낙상 판별용 핵심 물리량(Feature) 추출
* **목표:** 3단계에서 평활화된 각 필터별 좌표 데이터를 활용하여 낙상의 핵심 지표인 상체 중심점(HSSC)의 위치, 속도, 가속도 3가지 피처를 계산합니다.
* **추출 피처:** 1. `HSSC_Y`: 상체 중심점의 수직 위치
  2. `VHSSC`: 상체 중심점의 수직 하강 속도 (1차 미분)
  3. `AHSSC`: 상체 중심점의 수직 하강 가속도 (2차 미분)
* **출력:** 기존 데이터에 3개 열이 추가된 `step4_features_{filter_name}.csv`

In [70]:
print("Step 4: 단일 파일 대상 핵심 피처(HSSC_X, HSSC_Y, VHSSC, AHSSC) 추출 시작...")

# =====================================================================
# [수정 포인트] 처리할 원본 파일명과 저장할 결과 파일명을 여기에 입력하세요.
# =====================================================================
input_csv = 'step3_filtered_1-Euro.csv'        # 읽어올 파일
output_csv = 'step4_features_single.csv'     # 저장할 파일

HSSC_INDICES = [0, 1, 2, 3, 4, 5, 6] # 코, 양눈, 양귀, 양어깨

try:
    # 데이터 로드
    df = pd.read_csv(input_csv)

    # 1. HSSC (상체 중심점 X, Y 좌표) 계산
    hssc_x_cols = [f'{KEYPOINT_NAMES[i]}_x' for i in HSSC_INDICES]
    hssc_y_cols = [f'{KEYPOINT_NAMES[i]}_y' for i in HSSC_INDICES]

    df['HSSC_X'] = df[hssc_x_cols].mean(axis=1, skipna=True)
    df['HSSC_Y'] = df[hssc_y_cols].mean(axis=1, skipna=True)

    dt = df['timestamp_sec'].diff()
    dt = dt.replace(0, 0.001).fillna(1/TARGET_FPS)

    # 2. VHSSC (수직 하강 속도) 계산
    df['VHSSC'] = df['HSSC_Y'].diff() / dt
    df['VHSSC'] = df['VHSSC'].fillna(0) # 첫 행 초기화

    # 3. AHSSC (수직 하강 가속도) 계산
    df['AHSSC'] = df['VHSSC'].diff() / dt
    df['AHSSC'] = df['AHSSC'].fillna(0) # 첫 행 초기화

    df.replace([np.inf, -np.inf], np.nan, inplace=True)
    df['VHSSC'] = df['VHSSC'].interpolate(method='linear').fillna(0)
    df['AHSSC'] = df['AHSSC'].interpolate(method='linear').fillna(0)

    # 결과 저장
    df.to_csv(output_csv, index=False)
    print(f" -> '{output_csv}' 생성 완료. (HSSC_X, HSSC_Y, VHSSC, AHSSC 피처 추가됨)")
    print("Step 4 완료: 피처 추출이 성공적으로 적용되었습니다.")

except FileNotFoundError:
    print(f"❌ 오류: '{input_csv}' 파일을 찾을 수 없습니다. 경로와 파일 이름을 다시 확인해주세요.")

Step 4: 단일 파일 대상 핵심 피처(HSSC_X, HSSC_Y, VHSSC, AHSSC) 추출 시작...
 -> 'step4_features_single.csv' 생성 완료. (HSSC_X, HSSC_Y, VHSSC, AHSSC 피처 추가됨)
Step 4 완료: 피처 추출이 성공적으로 적용되었습니다.
